# ML-07 — Baseline Action Score (Week 4)

Lane: **Refresh / Content Opportunity Scoring** — confirmed this week, not switched.

Three things in this notebook, in order:

1. **Two signal checks** — one bucket table each with `n` printed, and a one-word verdict. At least one is tied to a real FlyRank flag.
2. **One rule, encoded live** — a transparent score, a single reason code, and an action label; the ranked queue is written to `work/outputs/baseline_action_score.csv`.
3. **A top-10 review** — one line per row: the action, why it's there, and what would make it wrong.

The rule uses only signals that are knowable at decision time (`impressions_90d`, `ctr`, `days_since_last_update`). It never touches `trend_direction`, `trend_pct`, or the decline label — the label is used only to *measure* verdicts and precision at the end, never to build the score.


## 1. My rule and its reason codes

The two signal checks live here first, because the rule must follow the evidence — not the other way around. Each check prints its bucket table with `n` and a one-word verdict: **CONFIRMED / OPPOSITE / MIXED / FALSE**.

### Signal check #1 — staleness behind the refresh flags (flag-linked)

**Claim:** a page that has not been updated in a long time is more likely to be declining — the assumption underneath FlyRank's refresh flags.

**Test:** bucket pages by `days_since_last_update` and compare the observed decline rate (`trend_direction == "down"`) across buckets, on the full 30,000-row starter slice.


In [1]:
# ---- Setup: find the repo root and load the starter slice ----
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(os.getcwd())
while not (ROOT / "data" / "raw" / "content_refresh_anonymized.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

df = pd.read_csv(ROOT / "data" / "raw" / "content_refresh_anonymized.csv")

# Label for MEASUREMENT ONLY — never a feature input. trend_direction / trend_pct stay out of the rule.
df["declining"] = (df["trend_direction"].astype(str).str.lower() == "down").astype(int)

print(f"loaded {len(df):,} rows from {ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv'}")

# ---- Signal check #1: staleness vs decline ----
bins = [0, 30, 90, 180, 365, 10**9]
labels = ["0-30 days", "31-90 days", "91-180 days", "181-365 days", "365+ days"]
df["staleness_bucket"] = pd.cut(df["days_since_last_update"], bins=bins, labels=labels, right=False)

table1 = (
    df.groupby("staleness_bucket", observed=True)
    .agg(
        n=("content_id", "size"),
        decline_rate_pct=("declining", lambda s: round(100 * s.mean(), 1)),
        median_impressions=("impressions_90d", "median"),
    )
    .reset_index()
)
table1["note"] = table1["n"].apply(lambda n: "" if n >= 50 else "insufficient data (n<50)")
print(table1.to_string(index=False))

print(f"\nBase rate (all pages): {100 * df['declining'].mean():.1f}% declining, n={len(df):,}")
print("VERDICT: MIXED — see the markdown cell below.")


loaded 30,000 rows from /Users/egealgel/Documents/FlyRankAI/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv
staleness_bucket     n  decline_rate_pct  median_impressions                     note
       0-30 days 20480              51.1               470.0                         
      31-90 days   175              58.9               510.0                         
     91-180 days  9171              61.1              1692.0                         
    181-365 days   169              46.7                16.0                         
       365+ days     5              60.0                 2.0 insufficient data (n<50)

Base rate (all pages): 54.2% declining, n=30,000
VERDICT: MIXED — see the markdown cell below.


**Verdict: MIXED.**

The refresh flag's core region is supported: the decline rate climbs from **51.1%** (0–30 days, n=20,480) to **58.9%** (31–90 days, n=175) to **61.1%** (91–180 days, n=9,171). But the signal does *not* keep rising past ~180 days — it reverses to **46.7%** (181–365 days, n=169), and the 365+ bucket (n=5) is below the sample-size floor and ignored.

What this means for the rule: staleness is worth using as a **gate** (≥90 days), but *not* as a linear "more stale = more urgent" score — the 181+ pages are mostly dead (median 16 impressions) and too far gone to be "declining" on a 30-day trend. That check just saved the rule from over-weighting the longest-stale pages.


### Signal check #2 — CTR weakness behind the CTR-fix logic (flag-linked)

**Claim:** among visible pages, a lower click-through rate is associated with a higher decline rate — the assumption underneath FlyRank's CTR-fix logic.

**Test:** on a defined slice (pages with real visibility: `impressions_90d >= 100` AND a measured position `avg_position > 0`), bucket by `ctr` (remember the gotcha: `ctr` is ×100, so `0.5` means 0.5%) and compare observed decline rates.


In [2]:
# ---- Signal check #2: CTR vs decline (visible pages only) ----
visible = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)].copy()

ctr_bins = [-1, 0.1, 0.5, 1.0, 2.0, 5.0, 10**9]
ctr_labels = ["<0.1%", "0.1-0.5%", "0.5-1.0%", "1.0-2.0%", "2.0-5.0%", "5%+"]
visible["ctr_bucket"] = pd.cut(visible["ctr"], bins=ctr_bins, labels=ctr_labels, right=False)

table2 = (
    visible.groupby("ctr_bucket", observed=True)
    .agg(
        n=("content_id", "size"),
        decline_rate_pct=("declining", lambda s: round(100 * s.mean(), 1)),
        median_impressions=("impressions_90d", "median"),
        median_position=("avg_position", "median"),
    )
    .reset_index()
)
table2["note"] = table2["n"].apply(lambda n: "" if n >= 50 else "insufficient data (n<50)")
print(f"slice: impressions_90d >= 100 AND avg_position > 0  (n={len(visible):,})")
print(table2.to_string(index=False))
print("\nVERDICT: CONFIRMED — see the markdown cell below.")


slice: impressions_90d >= 100 AND avg_position > 0  (n=22,006)
ctr_bucket    n  decline_rate_pct  median_impressions  median_position                     note
     <0.1% 8928              64.6               771.5             18.8                         
  0.1-0.5% 9686              58.5              2922.5             10.3                         
  0.5-1.0% 2541              51.7              2672.0              8.5                         
  1.0-2.0%  730              48.1              2015.0              8.0                         
  2.0-5.0%  106              38.7               328.0              7.1                         
       5%+   15              53.3               138.0              4.8 insufficient data (n<50)

VERDICT: CONFIRMED — see the markdown cell below.


**Verdict: CONFIRMED.**

The decline rate falls steadily as CTR rises: **64.6%** (<0.1%, n=8,928) → **58.5%** (0.1–0.5%, n=9,686) → **51.7%** (0.5–1.0%, n=2,541) → **48.1%** (1.0–2.0%, n=730) → **38.7%** (2.0–5.0%, n=106). The 5%+ bucket (n=15) is below the floor and ignored. Low CTR is a real, monotonic signal on visible pages.

### The rule these two checks support

Plain words: **"A page is worth a reviewer's time if it still has real search demand, it has not been updated in at least 90 days, and its click-through rate is weak."**

Encoded, all three parts transparent:

- **Score** `= stale × demand × ctr_weak`, range [0, 1]:
  - `stale = (days_since_last_update >= 90)` — a 0/1 gate (check #1: staleness is a gate, not a linear reward)
  - `demand = percentile_rank(log1p(impressions_90d))` — 0–1, how much traffic is at stake
  - `ctr_weak = 1 - min(ctr, 2) / 2` — 0–1, with `ctr` in ×100 points (0 → 1.0, 0.5 → 0.75, 2+ → 0)
- **Reason code (one):** `stale_low_ctr_demand`
- **Action label:** `refresh` when the score is > 0, else `monitor`

Rows below the staleness gate score 0 and carry the sentinel reason `not_flagged` / action `monitor` — they are excluded from the action list, not scored by a second rule.


## 2. Build the ranked queue (writes the CSV)

The cell below codes the score exactly as written above, ranks every row, attaches the reason code and action label, and writes `work/outputs/baseline_action_score.csv`. It also writes a metrics receipt JSON (committed, not gitignored) and prints the top-20 with precision@50 against the label — measured, never used as an input.


In [3]:
# ---- The rule, encoded live ----
def percentile_rank(series: pd.Series) -> pd.Series:
    return series.rank(method="average", pct=True).fillna(0)

stale = (df["days_since_last_update"] >= 90).astype(int)
demand = percentile_rank(np.log1p(df["impressions_90d"]))
ctr_weak = 1 - df["ctr"].clip(lower=0, upper=2) / 2.0

df["baseline_action_score"] = (stale * demand * ctr_weak).round(6)
df["reason_code"] = np.where(df["baseline_action_score"] > 0, "stale_low_ctr_demand", "not_flagged")
df["action_label"] = np.where(df["baseline_action_score"] > 0, "refresh", "monitor")
df["baseline_rank"] = df["baseline_action_score"].rank(method="first", ascending=False).astype(int)

output_columns = [
    "baseline_rank",
    "content_id",
    "client_id",
    "baseline_action_score",
    "reason_code",
    "action_label",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "days_since_last_update",
    "content_age_days",
    "ctr",
    "avg_position",
    "trend_direction",
]

queue = df.sort_values("baseline_rank")[output_columns]

OUT_DIR = ROOT / "work" / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
csv_path = OUT_DIR / "baseline_action_score.csv"
queue.to_csv(csv_path, index=False)
print(f"wrote {csv_path} ({len(queue):,} rows)")

# ---- Honest evaluation: precision@50 vs the label (measured only, never a feature) ----
declining_map = df.set_index("content_id")["declining"]
p50 = float(queue.head(50)["content_id"].map(declining_map).mean())
base_rate = float(df["declining"].mean())
print(f"\nprecision@50 (vs decline label): {p50:.3f}")
print(f"base rate (label mean):          {base_rate:.3f}")
print(f"lift over base rate:             {(p50 / base_rate - 1) * 100:+.1f}%")

# ---- Metrics receipt (committed to git; the CSV is regenerated and stays out) ----
metrics = {
    "rows": int(len(queue)),
    "flagged_rows": int((queue["action_label"] == "refresh").sum()),
    "precision_at_50_label": round(p50, 4),
    "base_rate_label": round(base_rate, 4),
    "score_formula": {
        "stale_gate": "days_since_last_update >= 90 (0/1)",
        "demand": "percentile_rank(log1p(impressions_90d))",
        "ctr_weak": "1 - min(ctr, 2) / 2",
        "score": "stale * demand * ctr_weak",
    },
    "reason_code": "stale_low_ctr_demand",
    "action_label": "refresh",
    "signal_verdicts": {
        "staleness_vs_decline": "MIXED",
        "ctr_vs_decline": "CONFIRMED",
    },
    "leakage_check": {
        "features_used": ["impressions_90d", "ctr", "days_since_last_update"],
        "label_used_as_feature": False,
        "trend_columns_used": False,
        "product_flags_used": False,
    },
}
metrics_path = OUT_DIR / "baseline_action_score_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2))
print(f"wrote {metrics_path}")

# ---- Top-20 preview (context for the top-10 review below) ----
print("\nTop 20 of the ranked queue:")
print(queue.head(20).to_string(index=False))


wrote /Users/egealgel/Documents/FlyRankAI/flyrank-ml-internship-starter/work/outputs/baseline_action_score.csv (30,000 rows)

precision@50 (vs decline label): 0.660
base rate (label mean):          0.542
lift over base rate:             +21.8%
wrote /Users/egealgel/Documents/FlyRankAI/flyrank-ml-internship-starter/work/outputs/baseline_action_score_metrics.json

Top 20 of the ranked queue:
 baseline_rank           content_id         client_id  baseline_action_score          reason_code action_label  impressions_90d  clicks_90d  sessions_90d  days_since_last_update  content_age_days  ctr  avg_position trend_direction
             1 content_c8e9d6ab9013 client_19581e27de               0.999067 stale_low_ctr_demand      refresh           208678           0             6                     104               362 0.00           9.7            down
             2 content_fb4bf6555c79 client_6208ef0f77               0.992367 stale_low_ctr_demand      refresh            84093           3      

## 3. Top-10 review

For each of the top ten, one line: the action, why it's there, and what would make it wrong. The cell below prints the same ten rows from the executed run so the review is grounded in the real numbers.


In [4]:
review_cols = [
    "baseline_rank",
    "content_id",
    "action_label",
    "reason_code",
    "baseline_action_score",
    "impressions_90d",
    "clicks_90d",
    "days_since_last_update",
    "content_age_days",
    "ctr",
    "avg_position",
    "trend_direction",
]
print(queue.head(10)[review_cols].to_string(index=False))


 baseline_rank           content_id action_label          reason_code  baseline_action_score  impressions_90d  clicks_90d  days_since_last_update  content_age_days  ctr  avg_position trend_direction
             1 content_c8e9d6ab9013      refresh stale_low_ctr_demand               0.999067           208678           0                     104               362 0.00           9.7            down
             2 content_fb4bf6555c79      refresh stale_low_ctr_demand               0.992367            84093           3                     104               299 0.00          45.6            down
             3 content_e752a4e03dd3      refresh stale_low_ctr_demand               0.992015           130892          15                     104               287 0.01          23.9            down
             4 content_54baba704595      refresh stale_low_ctr_demand               0.991982           130617           8                     104               286 0.01          47.0            down
     

1. **refresh** — `content_c8e9d6ab9013`: 208,678 impressions, 0 clicks, 104 days stale, 362 days old, position 9.7. Highest demand in the queue with a near-zero CTR, so it is the most traffic at stake. *Wrong if* those impressions come from a snippet/carousel where users don't click by design — then refreshing the body won't lift clicks.
2. **refresh** — `content_fb4bf6555c79`: 84,093 impressions, 3 clicks, position 45.6. Stale, visible, and deep — a classic refresh candidate. *Wrong if* position 45.6 means the page already lost the query it ranked for, and a rewrite targets a query that no longer exists.
3. **refresh** — `content_e752a4e03dd3`: 130,892 impressions, 15 clicks, position 23.9. Huge demand, tiny CTR. *Wrong if* the page is a comparison/list surfaced as a rich result, where the fix is structured data rather than a rewrite.
4. **refresh** — `content_54baba704595`: 130,617 impressions, 8 clicks, position 47.0. Same pattern — stale + visible + near-zero CTR. *Wrong if* the low CTR is a title/meta snippet problem the client can't edit, not a content-staleness problem.
5. **refresh** — `content_124763d39ca5`: 129,803 impressions, 17 clicks, position 33.2. Deep position with real impressions. *Wrong if* the query intent shifted (the page now ranks for a different, wrong query) — a refresh won't fix a mismatch.
6. **refresh** — `content_4a6607efcb46`: 128,068 impressions, 17 clicks, position **2.2**, but `trend_direction = up`. A top-3 page with 0.01% CTR is the anomaly: the rule flags it on CTR weakness even though it is not declining. *Wrong if* the page sits in a featured-snippet/rich-result slot where CTR is structurally near-zero — then a page refresh is the wrong lever.
7. **refresh** — `content_109f8f7c9d39`: 90,476 impressions, 6 clicks, position 54.4, `trend_direction = up`. *Wrong if* it is a PAA / related-questions impression source — no page edit changes those impressions.
8. **refresh** — `content_32cfb0b2fccf`: 89,361 impressions, 10 clicks, position 38.5. Deep and stale with real volume. *Wrong if* the page is being cannibalized by a newer page targeting the same keyword — the fix is consolidation, not refresh.
9. **refresh** — `content_8b36799b7e44`: 141,400 impressions, 23 clicks, position 32.0. Highest-impression row in the top 10, stale and underperforming. *Wrong if* the impressions are inflated by a seasonal spike that has already passed — the trailing-90-day window may overstate ongoing demand.
10. **refresh** — `content_f8de7d4cee60`: 89,803 impressions, 21 clicks, position 30.8. Stale, visible, mid-CTR. *Wrong if* the page is only 144 days old and still on a normal decay curve, where a small update — not a full refresh — is the right action.


## 4. Weak picks + leakage check

The rule is honest about where it's weak. Two weaknesses stand out, and the leakage check confirms the score uses only decision-time signals.


In [5]:
# ---- Weak pick #1: client concentration in the top of the queue ----
top50 = queue.head(50)
print("clients represented in the top 50:", top50["client_id"].nunique())
print(top50["client_id"].value_counts().head().to_string())
top_client_share = float(top50["client_id"].value_counts().max() / 50)
print(f"\nlargest client share of the top 50: {top_client_share:.0%}")

# ---- Weak pick #2: non-declining rows in the top 50 ----
declining_map = df.set_index("content_id")["declining"]
top50_declining = top50["content_id"].map(declining_map)
print(f"non-declining rows in the top 50: {int((top50_declining == 0).sum())} of 50")
print(f"zero-click rows in the top 50:     {int((top50['clicks_90d'] == 0).sum())} of 50")

# ---- Leakage check: prove the score never saw the label, a trend column, or a product flag ----
features_used = {"impressions_90d", "ctr", "days_since_last_update"}
label_derived = {"trend_direction", "trend_pct"}
assert not (label_derived & features_used), "label-derived column leaked into the feature set!"
print("\nleakage check: score inputs =", sorted(features_used))
print("label / trend columns used as inputs: none (trend_direction, trend_pct excluded)")
print("product flags used: none (no health_score / quick_win / flag columns are inputs)")


clients represented in the top 50: 4
client_id
client_6208ef0f77    35
client_19581e27de    13
client_3fdba35f04     1
client_624b60c58c     1

largest client share of the top 50: 70%
non-declining rows in the top 50: 17 of 50
zero-click rows in the top 50:     1 of 50

leakage check: score inputs = ['ctr', 'days_since_last_update', 'impressions_90d']
label / trend columns used as inputs: none (trend_direction, trend_pct excluded)
product flags used: none (no health_score / quick_win / flag columns are inputs)


**Weak picks, read honestly.**

1. **Client concentration.** The top 50 come from 4 clients, and ~70% from a single client whose pages all went ~104 days without an update. The rule is doing its job for that client, but a cross-client queue that one account floods is not a fair global priority list. A per-client ranking (or a client-normalized demand term) is the first improvement — deliberately left out of this baseline so the Week-5 model has an honest, simple target to beat.
2. **Top-3 / rich-result false positives.** Rows like #6 (position 2.2, 0.01% CTR, trending *up*) are flagged on CTR weakness even though they are not declining. The rule treats every low CTR as "needs refresh," but some low CTRs are structural (snippets, carousels, PAA) where a refresh is the wrong lever. This is exactly where a model that can see position tier and content type together should beat the rule.

The leakage check confirms the score is built only from `impressions_90d`, `ctr`, and `days_since_last_update` — no `trend_direction`, no `trend_pct`, no label, no product flags, and no future window. The label (`declining`) is used in sections 1–4 only to *measure* verdicts and precision, never to produce the score.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (executed via nbconvert)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit the repo URL on the card. Done.
